# 🧪 GIADA Task38 — Joint explicit-state ionic transition
Six paired GPU members; fixed Task37 quality thresholds. No autonomous compartment or speedup claim. The launcher must set GIADA_TASK38_REVISION to the full pushed commit before setup.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
REVISION=os.environ.get('GIADA_TASK38_REVISION','')
assert len(REVISION)==40 and all(c in '0123456789abcdef' for c in REVISION), 'Set GIADA_TASK38_REVISION to the full launch commit, not a branch.'
WORK=Path('/kaggle/working/giada_task38'); WORK.mkdir(exist_ok=True)
REPO=WORK/'giada'
if not REPO.exists(): subprocess.run(['git','clone','--depth','1','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','fetch','--depth','1','origin',REVISION],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==REVISION
OUTPUT=Path('/kaggle/working/artifacts/giada_task38_joint_transition')
assert not OUTPUT.exists(), f'Output exists: {OUTPUT}; use a clean run.'
print({'task':38,'revision':REVISION,'input_datasets_required':0})

In [ ]:
log=WORK/'process.log'
command=[sys.executable,'-u',str(REPO/'scripts/run_task38_joint_transition.py'),str(OUTPUT)]
with log.open('w') as saved:
    worker=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in worker.stdout:
        saved.write(line); saved.flush()
        if '[GIADA Task38]' in line: print(line.rstrip(),flush=True)
    exit_code=worker.wait()
OUTPUT.mkdir(parents=True,exist_ok=True); shutil.copy2(log,OUTPUT/'process.log')
report_path=OUTPUT/('final_report.json' if (OUTPUT/'final_report.json').exists() else 'failure_report.json')
if report_path.exists(): report=json.loads(report_path.read_text())
else:
    report={'valid':False,'technical_failure':True,'scientific_no_go':False,'exit_code':exit_code,'error':'Worker exited without report; see process.log'}
    report_path.write_text(json.dumps(report,indent=2))
print({k:report.get(k) for k in ('valid','decision','technical_failure','error','summaries')})
if exit_code: print('\n'.join(log.read_text().splitlines()[-18:]))
archive=shutil.make_archive('/kaggle/working/giada_task38_joint_transition','zip',OUTPUT)
print({'archive_ready':archive,'worker_exit_code':exit_code})

In [ ]:
# Optional browser download, using the project's tested Blob method.
import base64
from IPython.display import Javascript, display
encoded=base64.b64encode(Path(archive).read_bytes()).decode('ascii')
display(Javascript("const s=atob('"+encoded+"');const b=new Uint8Array(s.length);for(let i=0;i<s.length;i++)b[i]=s.charCodeAt(i);const u=URL.createObjectURL(new Blob([b],{type:'application/zip'}));const a=document.createElement('a');a.href=u;a.download='giada_task38_joint_transition.zip';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),60000);"))